# 06. Optional LoRA or Domain Adaptation

This optional notebook demonstrates a minimal PEFT/LoRA workflow after a retrieval baseline is already in place.

The framing here is intentionally conservative:
- do not assume full fine-tuning
- do not assume LoRA replaces retrieval
- evaluate style changes separately from actual task improvement

In small archive assistants, LoRA can be useful, but it can also create a false sense of progress if retrieval is still weak.


The workshop treats the system as retrieval-first:
- tokenization defines what the model sees
- retrieval quality often controls answer quality
- generators cannot reliably compensate for weak retrieval
- evaluation should include groundedness, citation, abstention, bilingual behavior, and community-review placeholders

Current notebook: `06_optional_lora_or_domain_adaptation.ipynb`

This notebook explores lightweight adaptation after a retrieval baseline exists, with LoRA positioned as a complement to retrieval rather than a substitute for it.

This notebook is optional and should usually be attempted after retrieval baselines are clear.

Workshop sequence:
1. `01_tokenization_playground.ipynb`
2. `02_embeddings_and_similarity.ipynb`
3. `03_retriever_benchmarking_for_rag.ipynb`
4. `04_llm_comparison_in_same_rag_pipeline.ipynb`
5. `05_evaluating_rag_systems.ipynb`
6. `06_optional_lora_or_domain_adaptation.ipynb` (optional)


## Quick concept refresher

- Tokenization turns text into units that models can process.
- Retrieval chooses which passages are available as evidence.
- The retriever selects context; the generator turns context into an answer.
- Evaluation in archive assistants is multi-layered because the system must be useful, grounded, reviewable, and appropriate.


In [ ]:
# Self-contained runtime setup for this notebook.
# Each notebook includes its own seed and device checks so it can run independently in Colab.

import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")
print(f"Detected device: {DEVICE}")
print(f"Seed set to: {SEED}")

NOTEBOOK_CONTEXT = {
    "seed": SEED,
    "device": DEVICE,
    "notebook": __name__ if "__name__" in globals() else "notebook",
    "framing": "retrieval-first archive assistant workshop",
}

NOTEBOOK_CONTEXT


In [ ]:
# In Colab, uncomment if needed.

# !pip -q install transformers datasets peft accelerate pandas

print("LoRA notebook dependencies are ready once the required packages are installed.")


## Practical note

This notebook is optional because even lightweight adaptation adds complexity:
- training loops take time
- GPUs help a lot
- evaluation becomes more important, not less

If you only have CPU access, you can still read through the workflow and reduce the training set and number of steps.


In [ ]:
# Imports.

import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


In [ ]:
# A tiny toy dataset.
# The examples are deliberately small because the goal is workflow literacy, not strong performance.

toy_examples = [
    {
        "context": "[D01] Restricted ceremonial recordings require community review before access.",
        "question": "Can this ceremonial recording be released immediately?",
        "target": "ABSTAIN. The evidence says community review is required before access. Citations: [D01]",
    },
    {
        "context": "[D02] Public catalog summaries should cite the original source note when describing place-name variants.",
        "question": "How should a public summary cite place-name variants?",
        "target": "Use the original source note when citing place-name variants. Citations: [D02]",
    },
    {
        "context": "[D03] Winter-only stories should not be surfaced outside the approved seasonal context.",
        "question": "Can the assistant quote this winter-only story in summer?",
        "target": "No. The story should not be surfaced outside the approved seasonal context. Citations: [D03]",
    },
    {
        "context": "[D04] Governance metadata must be checked before showing kinship-based access materials.",
        "question": "What should the system check before showing kinship-based materials?",
        "target": "It should check governance metadata first. Citations: [D04]",
    },
]

toy_df = pd.DataFrame(toy_examples)
toy_df


In [ ]:
# Choose a small seq2seq model that is realistic for demonstration.
# flan-t5-small is light enough for a workshop demo and works well with PEFT.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
base_model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name).to(DEVICE)

print("Loaded base model:", base_model_name)
print("Using device:", DEVICE)


In [ ]:
# Turn the toy examples into an instruction-style dataset.
# We keep the prompt format simple and explicit.

def format_prompt(example):
    return {
        "input_text": (
            "Use only the provided context. "
            "If the evidence is insufficient, answer ABSTAIN.\n\n"
            f"Context: {example['context']}\n"
            f"Question: {example['question']}"
        ),
        "target_text": example["target"],
    }


formatted_examples = [format_prompt(example) for example in toy_examples]
dataset = Dataset.from_list(formatted_examples)
dataset


In [ ]:
# Tokenize the dataset.
# This function is intentionally verbose so participants can modify lengths and formatting easily.

MAX_INPUT_LENGTH = 192
MAX_TARGET_LENGTH = 96


def tokenize_example(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_dataset = dataset.map(tokenize_example, batched=True, remove_columns=dataset.column_names)
tokenized_dataset


In [ ]:
# Configure LoRA.
# For T5-style attention modules, q and v are common lightweight targets.

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()


In [ ]:
# A very small training configuration.
# This is intentionally modest so the notebook remains demonstrative rather than expensive.

training_args = Seq2SeqTrainingArguments(
    output_dir="./lora_demo_outputs",
    learning_rate=5e-4,
    per_device_train_batch_size=2,
    num_train_epochs=8,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    fp16=torch.cuda.is_available(),
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=lora_model)

trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


In [ ]:
# Before training, compare base and LoRA-wrapped outputs on a held-out prompt.
# The LoRA model starts from the same base behavior before optimization.

test_prompt = (
    "Use only the provided context. If the evidence is insufficient, answer ABSTAIN.\n\n"
    "Context: [D05] Public summaries can describe restricted materials at a high level but should not quote them directly.\n"
    "Question: Can the assistant quote the restricted material directly?"
)


def generate_text(model, prompt, max_new_tokens=64):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {name: tensor.to(next(model.parameters()).device) for name, tensor in inputs.items()}
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print("Base model output before LoRA training:")
print(generate_text(base_model, test_prompt))

print("\nLoRA model output before training:")
print(generate_text(lora_model, test_prompt))


In [ ]:
# Train the LoRA adapters.
# On CPU this may be slow; on Colab GPU it should be manageable because the dataset is tiny.

trainer.train()


In [ ]:
# Compare behavior after adaptation.
# We are mostly looking for stylistic changes or slightly better task formatting.
# Strong performance gains would be surprising with such a tiny dataset.

print("Base model output after LoRA training step (base weights unchanged):")
print(generate_text(base_model, test_prompt))

print("\nAdapted LoRA model output:")
print(generate_text(lora_model, test_prompt))


## Risks to discuss

- Overfitting: the model may memorize the tiny examples rather than learn a robust behavior
- Memorization: sensitive phrasing or archival conventions may be reproduced too literally
- Catastrophic forgetting: less likely with LoRA than full fine-tuning, but still worth monitoring
- False progress: style can improve while retrieval remains the real bottleneck

A good rule of thumb is to compare any adaptation against a strong retrieval-first baseline before claiming success.


In [ ]:
# Exercise ideas:
# 1. Add more examples that require abstention.
# 2. Add bilingual prompts and see whether the adaptation helps or just changes style.
# 3. Lower num_train_epochs and compare whether the behavior change is stable.

print("Suggested exercise: add one more governance-focused example and rerun tokenization + training.")
